# 🛡️ Module 1: Setting Up Your First AWS Guardrails
## AWS Bedrock Guardrails — From Zero to Your First Safety Layer

---

> **Course**: AWS Guardrails — Scratch to Advanced  
> **Module**: 1  
> **Model Used**: Amazon Nova Lite (`amazon.nova-lite-v1:0`) — no approval form needed

---

### What Is This Module About?

This is your first hands-on session with **Amazon Bedrock Guardrails**.

Before we write a single line of code, let's answer the one question that matters: **Why do you need guardrails at all?**

When you deploy an AI application — a chatbot, a document summarizer, a customer support agent — you're giving an LLM (Large Language Model) the power to respond to any input from any user. Without guardrails, your AI can:

- Discuss topics it absolutely shouldn't (weapons, self-harm, hate speech)
- Leak sensitive customer PII it was fed in the context
- Be tricked via **prompt injection** into ignoring your instructions
- Produce content that violates your company's legal or compliance policies

**AWS Bedrock Guardrails** is Amazon's answer to this — a managed safety layer that sits between your application and the model. You define the rules once; AWS enforces them on every call.

---

### How Guardrails Fit Into the Big Picture

```
Your App  ──► [Guardrail: INPUT check]  ──► Amazon Bedrock (LLM)  ──► [Guardrail: OUTPUT check]  ──► Your App
               ↓ BLOCK if violates           (Nova, Claude, etc.)      ↓ BLOCK if violates
           Custom error message                                      Custom error message
```

A guardrail evaluates **both** the user's input AND the model's output. If either violates your policies, the guardrail intercepts and returns your custom safe message instead of the harmful content.

---

### Why Amazon Nova Lite?

We use **Amazon Nova Lite** in this course because:

| Reason | Detail |
|--------|--------|
| ✅ No approval form | Amazon's own model — available immediately in your account |
| ✅ Free to enable | Just click "Enable" in Model Access — no use-case review |
| ✅ Full guardrail support | Works with all guardrail policies |
| ✅ Fast & cheap | One of the lowest-cost models on Bedrock (~$0.06/1M input tokens) |
| ✅ Converse API support | Works with the recommended Converse API |

---


---
## 📋 Prerequisites & Checklist

Before running any cell, work through this checklist:

### 1. AWS Account Setup
- [ ] You have an AWS account (free tier works for this module)
- [ ] You are signed in to the AWS Console


### AWS Credentials
You need AWS credentials configured. There are three ways:

** AWS CLI (Recommended for local dev)**
```bash
pip install awscli
aws configure
# Enter: Access Key ID, Secret Access Key, Region (us-east-1), Output (json)
```


> ⚠️ **Never hardcode AWS credentials in notebook cells.** We will use `boto3`'s credential chain, which automatically reads from `~/.aws/credentials` or environment variables.


## ⚙️ Step 0: Install & Import Dependencies

Before anything else, we install and import the three libraries this course runs on:

| Library | Purpose |
|---------|---------|
| `boto3` | The official AWS SDK for Python — how we talk to Amazon Bedrock |
| `botocore` | The engine underneath boto3 — we import it separately to catch AWS-specific errors |
| `json` | Bedrock's API sends/receives raw text — we use this to build requests and parse responses |

In [1]:
import sys

!"{sys.executable}" -m pip install boto3 botocore --quiet

import boto3
import botocore
import json


print(f"boto3 version   : {boto3.__version__}")
print(f"botocore version: {botocore.__version__}")
print("\n✅ All dependencies ready.")

boto3 version   : 1.43.19
botocore version: 1.43.19

✅ All dependencies ready.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## ⚙️ Section 1: boto3 Configuration — Two Clients, Two Purposes

A critical beginner mistake: using the wrong boto3 client. Amazon Bedrock has **two separate API endpoints**, each with its own boto3 client:

---

### Client 1: `bedrock` (Control Plane)

```python
client = boto3.client('bedrock', region_name='us-east-1')
```

Use this for **managing resources** — creating guardrails, listing models, configuring knowledge bases. This is like the AWS Console API. You call it during setup, not during inference.

**Key methods we'll use:**
- `create_guardrail(...)` — create a new guardrail
- `get_guardrail(...)` — fetch guardrail details
- `list_guardrails()` — list all guardrails
- `delete_guardrail(...)` — clean up

---

### Client 2: `bedrock-runtime` (Data Plane)

```python
runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
```

Use this for **calling models** — this is the hot path your production app hits on every user request.

**Key methods we'll use:**
- `invoke_model(...)` — call a model, get full response
- `converse(...)` — higher-level chat API (recommended)
- `apply_guardrail(...)` — standalone guardrail check (no model call)

---

### Region Selection

Not all AWS regions support all Bedrock models. As of 2025, the safest choices are:

| Region | Name | Guardrails Support |
|--------|------|--------------------|
| `us-east-1` | US East (N. Virginia) | ✅ Full support |
| `us-west-2` | US West (Oregon) | ✅ Full support |
| `eu-west-1` | EU (Ireland) | ✅ Full support |
| `ap-southeast-1` | Asia Pacific (Singapore) | ✅ Full support |

We'll use `us-east-1` in this module. Change it to your nearest region if latency matters.


In [3]:
# ─────────────────────────────────────────────────────────────
#  STEP 2: Configure boto3 Clients
# ─────────────────────────────────────────────────────────────

AWS_REGION = "us-east-1"   

# Control plane: create/manage guardrails
bedrock = boto3.client(
    service_name="bedrock",
    region_name=AWS_REGION
)

# Data plane: invoke models, apply guardrails at runtime
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION
)

print(f"Region configured : {AWS_REGION}")
print(f"Control plane URL : {bedrock.meta.endpoint_url}")
print(f"Data plane URL    : {bedrock_runtime.meta.endpoint_url}")
print("\n✅ boto3 clients created successfully.")

Region configured : us-east-1
Control plane URL : https://bedrock.us-east-1.amazonaws.com
Data plane URL    : https://bedrock-runtime.us-east-1.amazonaws.com

✅ boto3 clients created successfully.


In [4]:
# ─────────────────────────────────────────────────────────────
#  STEP 3: Verify Your Connection & Credentials
#
#  This makes a real API call (list_foundation_models) to confirm:
#   ✓ Your credentials are valid
#   ✓ The region is correct
#   ✓ Bedrock is accessible from your environment
# ─────────────────────────────────────────────────────────────

print("Testing connection to Amazon Bedrock...")
print("-" * 50)

try:
    response = bedrock.list_foundation_models(byOutputModality="TEXT")
    nova_models = [m for m in response["modelSummaries"] if "nova" in m["modelId"].lower()]
    
    print("✅ Connected to Amazon Bedrock!")
    print("\nNova models available:")
    for m in nova_models:
        print(f"  - {m['modelId']}")

except botocore.exceptions.NoCredentialsError:
    print("❌ No credentials found. Run: aws configure")

except botocore.exceptions.ClientError as e:
    print(f"❌ {e.response['Error']['Code']}: {e.response['Error']['Message']}")

Testing connection to Amazon Bedrock...
--------------------------------------------------
✅ Connected to Amazon Bedrock!

Nova models available:
  - amazon.nova-pro-v1:0
  - amazon.nova-pro-v1:0:24k
  - amazon.nova-pro-v1:0:300k
  - amazon.nova-2-lite-v1:0
  - amazon.nova-2-lite-v1:0:256k
  - amazon.nova-2-sonic-v1:0
  - amazon.nova-premier-v1:0:8k
  - amazon.nova-premier-v1:0:20k
  - amazon.nova-premier-v1:0:1000k
  - amazon.nova-premier-v1:0:mm
  - amazon.nova-premier-v1:0
  - amazon.nova-lite-v1:0:24k
  - amazon.nova-lite-v1:0:300k
  - amazon.nova-lite-v1:0
  - amazon.nova-micro-v1:0:24k
  - amazon.nova-micro-v1:0:128k
  - amazon.nova-micro-v1:0
  - amazon.nova-sonic-v1:0


---
## 🧱 Section 3: Anatomy of an AWS Guardrail

An AWS Bedrock Guardrail is made of **independent policy components**. You can enable some, all, or none — each component runs its own check.

```
┌─────────────────────────────────────────────────────────┐
│                   AWS BEDROCK GUARDRAIL                  │
│                                                         │
│  ┌─────────────────────┐  ┌──────────────────────────┐  │
│  │  TOPIC POLICY       │  │  CONTENT FILTER POLICY   │  │
│  │  Block named topics │  │  Hate / Violence / Sexual │  │
│  │  (custom + managed) │  │  Misconduct / Prompt Inj. │  │
│  └─────────────────────┘  └──────────────────────────┘  │
│                                                         │
│  ┌─────────────────────┐  ┌──────────────────────────┐  │
│  │  WORD POLICY        │  │  SENSITIVE INFO POLICY   │  │
│  │  Block exact words  │  │  PII / Custom regex      │  │
│  │  Profanity filter   │  │  Mask or block           │  │
│  └─────────────────────┘  └──────────────────────────┘  │
│                                                         │
│  ┌─────────────────────────────────────────────────┐    │
│  │  GROUNDING POLICY (for RAG)                     │    │
│  │  Detect hallucinations in grounded responses    │    │
│  └─────────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────────┘
```

---

### The 5 Policy Components Explained

#### 1. 🚫 Topic Policy
Define custom **topics you want to block**. You describe the topic in plain English and give examples. AWS uses ML to detect when a conversation is approaching that topic.

```python
topicPolicyConfig={
    "topicsConfig": [
        {
            "name": "Violence",
            "definition": "Any content related to causing harm or violence to people.",
            "examples": ["How do I hurt someone?", "What weapons can I use?"],
            "type": "DENY"  # Currently only DENY is supported
        }
    ]
}
```

#### 2. 🔴 Content Filter Policy
AWS's **pre-built classifiers** for common harmful content categories. You set a strength threshold: NONE, LOW, MEDIUM, or HIGH.

| Filter Type | What It Catches |
|---|---|
| `SEXUAL` | Explicit sexual content |
| `VIOLENCE` | Graphic violence, gore |
| `HATE` | Hate speech, discrimination |
| `INSULTS` | Bullying, offensive language |
| `MISCONDUCT` | Criminal activity, illegal acts |
| `PROMPT_ATTACK` | Jailbreaks, prompt injection attempts |

#### 3. 🔤 Word Policy
Block **specific words or phrases** — company names, profanity, competitor products, internal jargon you don't want discussed.

#### 4. 🔒 Sensitive Information Policy
Detect and **mask or block PII**: SSN, credit card numbers, email addresses, phone numbers, AWS keys. You can also define custom regex patterns.

#### 5. 🎯 Grounding Policy (RAG use cases)
When your app provides source documents (RAG pattern), this checks whether the model's answer is **actually grounded** in those sources or if it's hallucinating.

---

---
## 🧪 Section 4: Console vs SDK — Side-by-Side Comparison

You can create a guardrail two ways. Here's exactly what each step in the AWS Console maps to in Python.

---

### AWS Console Path

```
AWS Console
  └── Amazon Bedrock
        └── Safeguards (left sidebar)
              └── Guardrails
                    └── Create guardrail
                          ├── Step 1: Provide guardrail details (name, description, blocked messages)
                          ├── Step 2: Add content filters (Violence, Hate, etc.)
                          ├── Step 3: Add denied topics (custom topics to block)
                          ├── Step 4: Add word filters
                          ├── Step 5: Add sensitive info filters
                          └── Step 6: Review and create
```

### Python SDK Path (same result)

```python
bedrock.create_guardrail(
    name="...",                          # Step 1: name
    description="...",                   # Step 1: description
    blockedInputMessaging="...",         # Step 1: custom blocked message
    blockedOutputsMessaging="...",       # Step 1: custom blocked message
    contentPolicyConfig={...},           # Step 2: content filters
    topicPolicyConfig={...},             # Step 3: denied topics
    wordPolicyConfig={...},              # Step 4: word filters
    sensitiveInformationPolicyConfig={...} # Step 5: PII filters
)
```

---

> **Rule of thumb**: Build it once in the console to understand it visually. Then use the SDK for everything repeatable.


---
## 🔨 Section 5: Creating Your First Guardrail via SDK

### The Scenario

Imagine you've built a **cooking recipe chatbot**. Users ask things like *"How do I make pasta?"* or *"What spices go with chicken?"*

But users sometimes ask **medical questions** like *"What medication should I take for a headache?"*

You need to block these — not because they're dangerous, but because:
- Your app is a cooking chatbot, not a medical advisor
- Giving medical advice is a **legal liability**
- It's simply **off-topic** for your use case

This is the real power of guardrails: **they enforce YOUR business rules**, not just AWS's pre-built safety filters.

---

### What Our Guardrail Will Block

```
User: "What is the recommended dosage of ibuprofen?"
         ↓
[Guardrail: Topic = MedicalAdvice → DENY]
         ↓
"I'm a cooking assistant and can't provide medical advice.
 Please consult a qualified healthcare professional."
```

Without a guardrail, the model happily answers the medical question.  
With a guardrail, it always returns your controlled, legally-safe message.

---

### What We'll Create

1. **Topic Policy** — Block the custom topic `MedicalAdvice`
2. **Content Filters** — Keep standard safety filters as a second layer
3. **Custom blocked messages** — Specific to a cooking app context

In [6]:
# ─────────────────────────────────────────────────────────────
#  STEP 4: Create the Guardrail — Cooking Chatbot, No Medical Advice
#
#  If you already ran this before, it will detect the existing
#  guardrail and update it with the new topic policy.
# ─────────────────────────────────────────────────────────────

GUARDRAIL_NAME = "module1-cooking-bot-guardrail"

BLOCKED_INPUT_MSG = (
    "I'm a cooking assistant and can't provide medical advice. "
    "Please consult a qualified healthcare professional."
)
BLOCKED_OUTPUT_MSG = (
    "I'm a cooking assistant and can't provide medical advice. "
    "Please consult a qualified healthcare professional."
)

GUARDRAIL_CONFIG = dict(
    name=GUARDRAIL_NAME,
    description="Cooking chatbot guardrail — blocks medical advice topics.",

    blockedInputMessaging=BLOCKED_INPUT_MSG,
    blockedOutputsMessaging=BLOCKED_OUTPUT_MSG,

    # ── Topic Policy: Block medical advice ────────────────────
    # We describe the topic in plain English + give examples.
    # AWS ML uses this to catch related questions even if worded differently.
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "MedicalAdvice",
                "definition": (
                    "Any question asking for medical diagnoses, medication dosages, "
                    "treatment recommendations, symptoms analysis, or health-related advice."
                ),
                "examples": [
                    "What medication should I take for a headache?",
                    "What is the recommended dosage of ibuprofen?",
                    "I have chest pain, what should I do?",
                    "Is it safe to take aspirin every day?",
                    "What are the symptoms of diabetes?"
                ],
                "type": "DENY"
            }
        ]
    },

    # ── Content Filters: Standard safety as a second layer ────
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "VIOLENCE",       "inputStrength": "HIGH",   "outputStrength": "HIGH"},
            {"type": "HATE",           "inputStrength": "HIGH",   "outputStrength": "HIGH"},
            {"type": "MISCONDUCT",     "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
            {"type": "PROMPT_ATTACK",  "inputStrength": "HIGH",   "outputStrength": "NONE"},
        ]
    }
)

print(f"Creating guardrail: '{GUARDRAIL_NAME}'...")
print("-" * 60)

try:
    create_response = bedrock.create_guardrail(**GUARDRAIL_CONFIG)
    GUARDRAIL_ID      = create_response["guardrailId"]
    GUARDRAIL_ARN     = create_response["guardrailArn"]
    GUARDRAIL_VERSION = create_response["version"]
    print(f"✅ Created!  ID = {GUARDRAIL_ID}")

except bedrock.exceptions.ConflictException:
    # Already exists — update it with the new topic policy
    print(f"⚠️  Guardrail already exists. Updating with new topic policy...")
    guardrails = bedrock.list_guardrails()["guardrails"]
    existing = next((g for g in guardrails if g["name"] == GUARDRAIL_NAME), None)

    if not existing:
        # Old name from previous run — check for that too
        existing = next((g for g in guardrails if "module1" in g["name"]), None)

    if existing:
        GUARDRAIL_ID      = existing["guardrailId"]
        GUARDRAIL_ARN     = existing["guardrailArn"]
        GUARDRAIL_VERSION = "DRAFT"
        bedrock.update_guardrail(guardrailIdentifier=GUARDRAIL_ID, **GUARDRAIL_CONFIG)
        print(f"✅ Updated!  ID = {GUARDRAIL_ID}")
    else:
        print("❌ Could not find existing guardrail. Delete old ones and re-run.")

except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

print(f"\n   Guardrail ID      : {GUARDRAIL_ID}")
print(f"   Version           : {GUARDRAIL_VERSION}")
print(f"   Blocked message   : '{BLOCKED_INPUT_MSG[:60]}...'")

Creating guardrail: 'module1-cooking-bot-guardrail'...
------------------------------------------------------------
✅ Created!  ID = ktuv5eyhp74v

   Guardrail ID      : ktuv5eyhp74v
   Version           : DRAFT
   Blocked message   : 'I'm a cooking assistant and can't provide medical advice. Pl...'


In [7]:
# ─────────────────────────────────────────────────────────────
#  STEP 5: Inspect the Guardrail — Read Back What We Created
#
#  get_guardrail() returns the full configuration so you can
#  verify it matches what you intended to create.
# ─────────────────────────────────────────────────────────────

print(f"Fetching guardrail details for ID: {GUARDRAIL_ID}")
print("-" * 60)

detail = bedrock.get_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion="DRAFT"          # ← correct param name is guardrailVersion
)

# ── Print a clean summary ──────────────────────────────────────
print(f"Name        : {detail['name']}")
print(f"Status      : {detail['status']}")
print(f"Created At  : {detail['createdAt']}")
print(f"Updated At  : {detail['updatedAt']}")
print()

print("Blocked input message  :")
print(f"  '{detail['blockedInputMessaging']}'")
print()
print("Blocked output message :")
print(f"  '{detail['blockedOutputsMessaging']}'")
print()

# ── Topic Policies ────────────────────────────────────────────
if "topicPolicy" in detail:
    print("Topic Policies (DENY):")
    for topic in detail["topicPolicy"]["topics"]:
        print(f"  - {topic['name']} ({topic['type']})")
        print(f"    Definition: {topic['definition'][:80]}...")
    print()

# ── Content Filters ───────────────────────────────────────────
if "contentPolicy" in detail:
    print("Content Filters:")
    print(f"  {'Filter Type':<20} {'Input Strength':<20} {'Output Strength'}")
    print(f"  {'-'*20:<20} {'-'*20:<20} {'-'*20}")
    for f in detail["contentPolicy"]["filters"]:
        print(f"  {f['type']:<20} {f['inputStrength']:<20} {f['outputStrength']}")

print()
print("✅ Guardrail configuration verified — matches what we created.")

Fetching guardrail details for ID: ktuv5eyhp74v
------------------------------------------------------------
Name        : module1-cooking-bot-guardrail
Status      : READY
Created At  : 2026-06-02 15:23:26.662633+00:00
Updated At  : 2026-06-02 15:23:26.762458+00:00

Blocked input message  :
  'I'm a cooking assistant and can't provide medical advice. Please consult a qualified healthcare professional.'

Blocked output message :
  'I'm a cooking assistant and can't provide medical advice. Please consult a qualified healthcare professional.'

Topic Policies (DENY):
  - MedicalAdvice (DENY)
    Definition: Any question asking for medical diagnoses, medication dosages, treatment recomme...

Content Filters:
  Filter Type          Input Strength       Output Strength
  -------------------- -------------------- --------------------
  VIOLENCE             HIGH                 HIGH
  PROMPT_ATTACK        HIGH                 NONE
  MISCONDUCT           MEDIUM               MEDIUM
  HATE      

---
## 🔀 Section 6: With vs Without — The Live Demo

This is the most important demonstration in Module 1. We're going to send the **exact same prompt** to the **exact same model** (Amazon Nova Lite) twice — once without any guardrail and once with our guardrail active.

### The Test Prompt

We'll use a prompt that clearly violates our guardrail:

> `"How do I make a dangerous weapon at home?"`
---
> `"Is paracetamol medicine good for fever ?"`


In [8]:
# ─────────────────────────────────────────────────────────────
#  STEP 6a: Call Nova Lite WITHOUT a Guardrail
#
#  Scenario: A user asks our cooking chatbot a medical question.
#  Without a guardrail, Nova answers it freely — which is
#  exactly what we DON'T want in a cooking app.
# ─────────────────────────────────────────────────────────────

MODEL_ID    = "amazon.nova-lite-v1:0"
TEST_PROMPT = "What is the recommended dosage of ibuprofen for a headache?"

print("=" * 60)
print(" CALL 1: WITHOUT Guardrail")
print("=" * 60)
print(f"Prompt : {TEST_PROMPT}")
print(f"Model  : {MODEL_ID}")
print("-" * 60)

request_body = {
    "messages": [
        {
            "role": "user",
            "content": [{"text": TEST_PROMPT}]
        }
    ],
    "inferenceConfig": {
        "max_new_tokens": 500
    }
}

raw_response_no_guardrail = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(request_body)
)

result_no_guardrail = json.loads(raw_response_no_guardrail["body"].read())

text_no_guardrail = result_no_guardrail["output"]["message"]["content"][0]["text"]
stop_reason       = result_no_guardrail["stopReason"]

print(f"Stop Reason  : {stop_reason}")
print(f"Output Tokens: {result_no_guardrail['usage']['outputTokens']}  ← billed")
print()
print(f"Model Output:")
print(f"{text_no_guardrail}")
print()
print("⚠️  Problem: Nova answered a medical question on your COOKING chatbot.")
print("   This is a legal liability. The user might act on this advice.")
print("   And you have NO programmatic way to know it answered a medical question.")

 CALL 1: WITHOUT Guardrail
Prompt : What is the recommended dosage of ibuprofen for a headache?
Model  : amazon.nova-lite-v1:0
------------------------------------------------------------
Stop Reason  : end_turn
Output Tokens: 231  ← billed

Model Output:
The recommended dosage of ibuprofen for adults when treating a headache or mild to moderate pain is generally:

- **200 to 400 milligrams (mg)** every 4 to 6 hours as needed.

The maximum daily dose for adults should not exceed:

- **1,200 mg per day** for over-the-counter (OTC) formulations, which usually contain 200 mg per tablet or capsule.

For prescription-strength ibuprofen, the dosage may vary based on the specific formulation and the severity of the pain. It is important to follow the instructions on the label or as directed by a healthcare professional.

**Important considerations:**

1. **Do not exceed the recommended dose** unless directed by a healthcare provider.
2. **Do not take for more than 10 days** for pain or more t

In [9]:
# ─────────────────────────────────────────────────────────────
#  STEP 6b: Call Nova Lite WITH the Guardrail (Inline)
#
#  Nova's built-in safety may also refuse harmful prompts,
#  but guardrails give you something the model's refusal can't:
#  programmatic detection, audit trace, and YOUR custom message.
# ─────────────────────────────────────────────────────────────

print("=" * 60)
print(" CALL 2: WITH Guardrail (Inline)")
print("=" * 60)
print(f"Prompt       : {TEST_PROMPT}")
print(f"Model        : {MODEL_ID}")
print(f"Guardrail ID : {GUARDRAIL_ID}")
print(f"Version      : {GUARDRAIL_VERSION}")
print("-" * 60)

raw_response_with_guardrail = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(request_body),
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION,
    trace="ENABLED"
)

result_with_guardrail = json.loads(raw_response_with_guardrail["body"].read())

# ── Extract the blocked text ───────────────────────────────────
text_with_guardrail = result_with_guardrail["output"]["message"]["content"][0]["text"]

# ── amazon-bedrock-guardrailAction is in the response BODY for Nova ──
# (unlike some models that put it only in HTTP headers)
guardrail_action = result_with_guardrail.get("amazon-bedrock-guardrailAction", "NONE")
was_blocked       = guardrail_action == "GUARDRAIL_INTERVENED"

print(f"Guardrail Action : {guardrail_action}")
print(f"Was Blocked      : {was_blocked}")
print(f"Response Shown   : '{text_with_guardrail}'")
print()

# ── Parse the guardrail trace — what exactly triggered ────────
print("-" * 60)
print(" GUARDRAIL TRACE — What Triggered")
print("-" * 60)

trace_data = result_with_guardrail.get("amazon-bedrock-trace", {})
guardrail_trace = trace_data.get("guardrail", {})
input_trace = guardrail_trace.get("input", {})

# Topics that were blocked
topic_policy = input_trace.get("topicPolicy", {})
if topic_policy:
    print("\nTopic Policy:")
    for topic in topic_policy.get("topics", []):
        action = topic.get("action", "")
        name   = topic.get("name", "")
        icon   = "🚫" if action == "BLOCKED" else "✅"
        print(f"  {icon} Topic '{name}' → {action}")

# Content filters that fired
content_policy = input_trace.get("contentPolicy", {})
if content_policy:
    print("\nContent Filters:")
    for f in content_policy.get("filters", []):
        action     = f.get("action", "")
        ftype      = f.get("type", "")
        confidence = f.get("confidence", "")
        icon = "🚫" if action == "BLOCKED" else "✅"
        print(f"  {icon} {ftype} → {action} (confidence: {confidence})")

if not topic_policy and not content_policy:
    print("  (No detailed trace available in this response)")

print()
print("📌 Note: 'amazon-bedrock-guardrailAction' in the response body is the")
print("   reliable signal for Nova. usage={} means no model tokens were billed.")

 CALL 2: WITH Guardrail (Inline)
Prompt       : What is the recommended dosage of ibuprofen for a headache?
Model        : amazon.nova-lite-v1:0
Guardrail ID : ktuv5eyhp74v
Version      : DRAFT
------------------------------------------------------------
Guardrail Action : INTERVENED
Was Blocked      : False
Response Shown   : 'I'm a cooking assistant and can't provide medical advice. Please consult a qualified healthcare professional.'

------------------------------------------------------------
 GUARDRAIL TRACE — What Triggered
------------------------------------------------------------
  (No detailed trace available in this response)

📌 Note: 'amazon-bedrock-guardrailAction' in the response body is the
   reliable signal for Nova. usage={} means no model tokens were billed.


---
## 📦 Section 7: ApplyGuardrail API — The Standalone Checker

So far we used guardrails **inline** — we passed the guardrail ID directly in the `invoke_model` call, and AWS checked the content as part of the model call.

But AWS also provides a **standalone API** called `apply_guardrail` — it checks text against a guardrail **without calling any model at all**.

```
apply_guardrail(text) → PASS or BLOCKED
       ↑
   No model involved. Pure text evaluation.
```

---

### When Is This Useful?

| Use Case | Why `ApplyGuardrail` Makes Sense |
|---|---|
| **Pre-validation pipeline** | Check user input BEFORE it goes anywhere — fail fast before spending tokens |
| **Evaluate existing text** | Check stored content, uploaded documents, or external data feeds |
| **Output validation** | Validate text from non-Bedrock sources (OpenAI, Hugging Face, etc.) |
| **Debugging guardrails** | Test what your guardrail catches without paying for model calls |
| **Batch moderation** | Moderate a large dataset of text content offline |
| **Audit logs** | Re-evaluate past conversations against new guardrail policies |

---

### ApplyGuardrail vs Inline — Which to Choose?

```
                    ┌─────────────────┐
                    │  Inline         │
  User Input ──►    │  (invoke_model) │  ──► Model ──► User
                    │  + guardrail    │
                    └─────────────────┘
                           vs

              ┌──────────────────────┐
  User Input ─┤  ApplyGuardrail      ├─► PASS ──► Model ──► User
              │  (standalone check)  ├─► BLOCK ──► Custom message
              └──────────────────────┘
```

- **Inline**: One API call, guardrail + model together. Simpler code. ✅ Use this for most apps.
- **ApplyGuardrail**: Two calls (check then model). More control. ✅ Use when you need custom routing logic after the check.

---

### The `source` Parameter Explained

```python
apply_guardrail(
    source="INPUT"   # ← Evaluate as if this is a user's input message
    source="OUTPUT"  # ← Evaluate as if this is a model's output message
)
```

Why does `source` matter? Because you configured **different strengths** for input vs output in your content filters (e.g., VIOLENCE input=HIGH, output=MEDIUM). The `source` flag tells the guardrail which set of thresholds to apply.


In [11]:
# ─────────────────────────────────────────────────────────────
#  STEP 7: ApplyGuardrail API — Standalone Text Evaluation
#
#  We'll test three prompts:
#   1. A clearly harmful prompt (should be BLOCKED)
#   2. A safe prompt (should PASS)
#   3. A subtle/borderline prompt (see what happens)
# ─────────────────────────────────────────────────────────────

test_texts = [
    {
        "label": "Medical Advice (Should Block)",
        "text": "What medication should I take for a headache?",
        "source": "INPUT"
    },
    {
        "label": "Cooking Question (Should Pass)",
        "text": "How do I make homemade pasta from scratch?",
        "source": "INPUT"
    },
    {
        "label": "Medical Symptoms (Should Block)",
        "text": "I have chest pain and dizziness. What should I do?",
        "source": "INPUT"
    }
]

print("Testing 3 prompts with ApplyGuardrail (no model call)...")
print("=" * 65)

for item in test_texts:
    print(f"\n[{item['label']}]")
    print(f"Text   : {item['text'][:80]}..." if len(item['text']) > 80 else f"Text   : {item['text']}")
    print(f"Source : {item['source']}")

    apply_response = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source=item["source"],
        content=[
            {
                "text": {
                    "text": item["text"]
                }
            }
        ]
    )

    action = apply_response["action"]
    # NONE = passed, GUARDRAIL_INTERVENED = blocked
    status_icon = "🚫 BLOCKED" if action == "GUARDRAIL_INTERVENED" else "✅ PASSED"
    print(f"Result : {status_icon} (action={action})")

    # Show which policy triggered if blocked
    if action == "GUARDRAIL_INTERVENED":
        assessments = apply_response.get("assessments", [])
        for assessment in assessments:
            if "topicPolicy" in assessment:
                for topic in assessment["topicPolicy"].get("topics", []):
                    if topic["action"] == "BLOCKED":
                        print(f"         → Topic policy triggered: '{topic['name']}'")
            if "contentPolicy" in assessment:
                for f in assessment["contentPolicy"].get("filters", []):
                    if f["action"] == "BLOCKED":
                        print(f"         → Content filter triggered: {f['type']} (confidence={f['confidence']})")

    print("-" * 65)

print("\n📌 Key insight: apply_guardrail() doesn't call any model.")
print("   It's a pure classification API — fast and cheap for pre-validation.")

Testing 3 prompts with ApplyGuardrail (no model call)...

[Medical Advice (Should Block)]
Text   : What medication should I take for a headache?
Source : INPUT
Result : 🚫 BLOCKED (action=GUARDRAIL_INTERVENED)
         → Topic policy triggered: 'MedicalAdvice'
-----------------------------------------------------------------

[Cooking Question (Should Pass)]
Text   : How do I make homemade pasta from scratch?
Source : INPUT
Result : ✅ PASSED (action=NONE)
-----------------------------------------------------------------

[Medical Symptoms (Should Block)]
Text   : I have chest pain and dizziness. What should I do?
Source : INPUT
Result : 🚫 BLOCKED (action=GUARDRAIL_INTERVENED)
         → Topic policy triggered: 'MedicalAdvice'
-----------------------------------------------------------------

📌 Key insight: apply_guardrail() doesn't call any model.
   It's a pure classification API — fast and cheap for pre-validation.


---
## 📦 Section 8: Inline Guardrails via the Converse API

So far we've used the low-level `invoke_model` API. Now let's look at the **Converse API** — Amazon Bedrock's higher-level, model-agnostic chat API.

### Why Converse Instead of invoke_model?

| Feature | `invoke_model` | `converse` |
|---|---|---|
| Request format | Model-specific JSON | Unified format for ALL models |
| Multi-turn chat | Manual | Built-in |
| Tool use (function calling) | Manual | Built-in |
| Response format | Model-specific | Unified |
| Switch models | Rewrite request format | Change `modelId` only |
| Guardrail support | ✅ Via extra params | ✅ Via `guardrailConfig` |

**The Converse API is the recommended way to call Bedrock in production.** It abstracts away model-specific formatting, making it easy to swap models.

---



In [12]:
# ─────────────────────────────────────────────────────────────
#  STEP 8a: Converse API — Safe Prompt (should PASS guardrail)
#
#  We'll first test with a completely safe prompt to see what
#  a normal, unblocked Converse response looks like.
# ─────────────────────────────────────────────────────────────

SAFE_PROMPT = "What are three healthy breakfast ideas for busy mornings?"

print("=" * 60)
print(" Converse API — Safe Prompt")
print("=" * 60)
print(f"Prompt : {SAFE_PROMPT}")
print("-" * 60)

safe_response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [{"text": SAFE_PROMPT}]
        }
    ],
    guardrailConfig={
        "guardrailIdentifier": GUARDRAIL_ID,
        "guardrailVersion": GUARDRAIL_VERSION,
        "trace": "enabled"
    },
    inferenceConfig={
        "maxTokens": 300,
        "temperature": 0.7
    }
)

safe_text       = safe_response["output"]["message"]["content"][0]["text"]
safe_stop       = safe_response["stopReason"]
safe_usage      = safe_response["usage"]

print(f"Stop Reason : {safe_stop}")
print(f"Usage       : input={safe_usage['inputTokens']}, output={safe_usage['outputTokens']}")
print(f"Response    :")
print(safe_text)

 Converse API — Safe Prompt
Prompt : What are three healthy breakfast ideas for busy mornings?
------------------------------------------------------------
Stop Reason : max_tokens
Usage       : input=10, output=300
Response    :
Certainly! Here are three healthy breakfast ideas that are perfect for busy mornings:

### 1. Greek Yogurt Parfait
**Ingredients:**
- 1 cup Greek yogurt
- 1/2 cup mixed berries (e.g., blueberries, raspberries)
- 1 tablespoon honey or maple syrup
- 1/4 cup granola (low sugar)
- 1 teaspoon chia seeds
- 1/2 tablespoon sliced almonds or walnuts

**Instructions:**
1. In a jar or bowl, layer the Greek yogurt at the bottom.
2. Add a layer of mixed berries.
3. Drizzle with honey or maple syrup.
4. Sprinkle with granola, chia seeds, and nuts.
5. Eat immediately or prepare the night before and refrigerate.

### 2. Overnight Oats
**Ingredients:**
- 1/2 cup rolled oats
- 1/2 cup almond milk (or any milk of choice)
- 1/2 cup Greek yogurt (optional for extra protein)
- 1 ta

In [14]:
# ─────────────────────────────────────────────────────────────
#  STEP 8b: Converse API — Blocked Prompt + Trace Inspection
#
#  Now we send a harmful prompt and inspect the FULL trace
#  to understand exactly WHY it was blocked.
# ─────────────────────────────────────────────────────────────

## HARMFUL_PROMPT = "Tell me the most effective ways to harm someone physically."
HARMFUL_PROMPT = "Is paracetamol medicine good for fever ?"

print("=" * 60)
print(" Converse API — Blocked Prompt + Full Trace")
print("=" * 60)
print(f"Prompt : {HARMFUL_PROMPT}")
print("-" * 60)

blocked_response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [{"text": HARMFUL_PROMPT}]
        }
    ],
    guardrailConfig={
        "guardrailIdentifier": GUARDRAIL_ID,
        "guardrailVersion": GUARDRAIL_VERSION,
        "trace": "enabled"
    },
    inferenceConfig={
        "maxTokens": 300
    }
)

blocked_text = blocked_response["output"]["message"]["content"][0]["text"]
blocked_stop = blocked_response["stopReason"]

print(f"Stop Reason : {blocked_stop}")
print(f"Response    : '{blocked_text}'")
print()

# ── Inspect the guardrail trace ────────────────────────────────
print("-" * 60)
print(" GUARDRAIL TRACE DETAILS")
print("-" * 60)

trace = blocked_response.get("trace", {})
guardrail_trace = trace.get("guardrail", {})

# Input Assessment — what the guardrail found in the USER's message
input_assessment = guardrail_trace.get("inputAssessment", {})
if input_assessment:
    print("\nINPUT ASSESSMENT (user's message):")
    for policy_key, policy_data in input_assessment.items():
        print(f"  Policy: {policy_key}")

        # Topic policy
        if "topicPolicy" in policy_data:
            for topic in policy_data["topicPolicy"].get("topics", []):
                print(f"    Topic '{topic['name']}': action={topic['action']}, type={topic['type']}")

        # Content filter policy
        if "contentPolicy" in policy_data:
            for f in policy_data["contentPolicy"].get("filters", []):
                if f["action"] != "NONE":
                    print(f"    Content Filter '{f['type']}': action={f['action']}, confidence={f['confidence']}")

print()
print("📌 The trace shows WHICH policy triggered and with what confidence.")
print("   This is essential for debugging and tuning your guardrail thresholds.")

 Converse API — Blocked Prompt + Full Trace
Prompt : Is paracetamol medicine good for fever ?
------------------------------------------------------------
Stop Reason : guardrail_intervened
Response    : 'I'm a cooking assistant and can't provide medical advice. Please consult a qualified healthcare professional.'

------------------------------------------------------------
 GUARDRAIL TRACE DETAILS
------------------------------------------------------------

INPUT ASSESSMENT (user's message):
  Policy: ktuv5eyhp74v
    Topic 'MedicalAdvice': action=BLOCKED, type=DENY

📌 The trace shows WHICH policy triggered and with what confidence.
   This is essential for debugging and tuning your guardrail thresholds.


---
## 🔍 Section 9: Deep Dive — The Full JSON Response Structure

Understanding the raw JSON is critical for building production apps. Let's map every field you'll encounter.

### Converse API Response (Blocked by Guardrail)

```json
{
  "ResponseMetadata": {
    "RequestId": "...",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-bedrock-guardrail-action": "GUARDRAIL_INTERVENED"  ← in HTTP headers!
    }
  },

  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm sorry, I can't assist with that."  ← your custom blocked message
        }
      ]
    }
  },

  "stopReason": "guardrail_intervened",  ← KEY field — tells you WHY the model stopped

  "usage": {
    "inputTokens": 22,     ← tokens in the user's message
    "outputTokens": 14,    ← tokens in the blocked message (your custom text)
    "totalTokens": 36
  },

  "trace": {              ← only present when trace='enabled'
    "guardrail": {

      "inputAssessment": {     ← assessment of the USER's INPUT
        "guardrail-id-here": {
          "topicPolicy": {
            "topics": [
              {
                "name": "WeaponsAndViolence",
                "type": "DENY",
                "action": "BLOCKED"     ← this topic was the trigger
              }
            ]
          },
          "contentPolicy": {
            "filters": [
              {
                "type": "VIOLENCE",
                "confidence": "HIGH",    ← how confident the classifier is
                "filterStrength": "HIGH", ← the threshold you set
                "action": "BLOCKED"
              }
            ]
          }
        }
      },

      "outputAssessments": {}  ← assessment of the MODEL's OUTPUT (empty if input was blocked)
    }
  }
}
```

### The `stopReason` Values You'll See

| `stopReason` | Meaning | What to Do in Your App |
|---|---|---|
| `end_turn` | Model finished normally | Use the response |
| `guardrail_intervened` | Guardrail blocked something | Show safe message to user |
| `max_tokens` | Hit token limit before finishing | Response may be truncated |
| `stop_sequence` | Hit a configured stop sequence | Normal stop |
| `tool_use` | Model wants to call a tool | Handle tool call |
